In [6]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import pandas as pd

In [ ]:
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
batch_size = 64
block_size = 256
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
max_iters = 5000

torch.manual_seed(42)

In [8]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [9]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [10]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [11]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [12]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


In [13]:
print(f"Cihaz: {device}, Vocab Size: {vocab_size}, n_embd: {n_embd}")

Cihaz: cuda, Vocab Size: 65, n_embd: 384


### Self-Attention Head Sınıfı

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # Attention score
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5) # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)

        v = self.value(x) # (B, T, head_size)
        out = wei @ v     # (B, T, head_size)
        return out

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

### Self-attention'lı Bigram Modeli

In [ ]:
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = MultiHeadAttention(n_head,n_embd // n_head)
        self.ln_f = nn.LayerNorm(n_embd) 
        self.lm_head = nn.Linear(n_embd, vocab_size)


    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, n_embd)
        x = tok_emb + pos_emb # (B, T, n_embd)

        # Self-attention uygula
        x = self.sa_head(x) # (B, T, n_embd)
        x = self.ln_f(x) # (B,T,C)
        # Logitleri üret
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

###  Modelin Eğitilmesi

In [19]:
torch.manual_seed(42)
model = BigramLanguageModel().to(device)
print(f"Toplam parameter: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"{iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

Toplam parameter: 739,265
0: train loss 4.2863, val loss 4.3023
300: train loss 2.2335, val loss 2.2642
600: train loss 1.9822, val loss 2.0762
900: train loss 1.9185, val loss 2.0300
1200: train loss 1.8828, val loss 2.0066
1500: train loss 1.8708, val loss 2.0225
1800: train loss 1.8525, val loss 1.9972
2100: train loss 1.8434, val loss 1.9938
2400: train loss 1.8374, val loss 2.0004
2700: train loss 1.8301, val loss 1.9889
3000: train loss 1.8238, val loss 1.9879
3300: train loss 1.8200, val loss 1.9961
3600: train loss 1.8152, val loss 1.9936
3900: train loss 1.8103, val loss 1.9938
4200: train loss 1.8146, val loss 1.9905
4500: train loss 1.8115, val loss 1.9945
4800: train loss 1.8118, val loss 1.9958
4999: train loss 1.8135, val loss 2.0014


### Tam Transformer Bloğu



In [ ]:
class TransformerLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x) 
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


In [21]:
torch.manual_seed(42)
model_ln = TransformerLanguageModel().to(device)
print(f"Toplam parameter: {sum(p.numel() for p in model_ln.parameters()):,}")

optimizer = torch.optim.AdamW(model_ln.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model_ln)
        print(f"{iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model_ln(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

final_losses_ln = estimate_loss(model_ln)
train_loss_ln = final_losses_ln['train']
val_loss_ln = final_losses_ln['val']


Toplam parameter: 10,788,929
0: train loss 4.3328, val loss 4.3372
300: train loss 2.0247, val loss 2.0881
600: train loss 1.5667, val loss 1.7466
900: train loss 1.3994, val loss 1.6270
1200: train loss 1.3061, val loss 1.5664
1500: train loss 1.2403, val loss 1.5324
1800: train loss 1.2022, val loss 1.5196
2100: train loss 1.1580, val loss 1.5099
2400: train loss 1.1227, val loss 1.5129
2700: train loss 1.0895, val loss 1.5037
3000: train loss 1.0545, val loss 1.5045
3300: train loss 1.0254, val loss 1.5107
3600: train loss 0.9958, val loss 1.5150
3900: train loss 0.9634, val loss 1.5222
4200: train loss 0.9311, val loss 1.5255
4500: train loss 0.9056, val loss 1.5459
4800: train loss 0.8734, val loss 1.5510
4999: train loss 0.8553, val loss 1.5586


### Tam Transformer ModelindenÖrnek Metin Üretimi

In [ ]:
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, loss = model(idx_cond)
        logits = logits[:, -1, :] # (B, C)
        probs = F.softmax(logits, dim=-1) # (B, C)
        idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
        idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    return idx

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text_ln = decode(generate(model_ln, context, max_new_tokens=500)[0].tolist())
print(generated_text_ln)



My lord, ill consider to Church off aloud deys.

CATESS OF YORK:
Which, we have fouged, God neither, for barned madam.

ABHORGORY:
Sweet prince, thus; 'tis the moonth before here.
These babes made us to their love.

BUSHY:
While Juliet, for we will; still all sent George begin,
The is as far as philosopher and Sundia.

THANRY BOLINGBROKE:
Faith, here comes the King Henry.

DUKE VINCENTIO:
A monthless things about my knows will at the Tower,
Abase me a harventure. This knowing men hath been an ho
